In [41]:
import torch
import torch.nn as nn

import bitsandbytes
from bitsandbytes.nn import Linear8bitLt

In [42]:
# define fp16 model
fp16_model = nn.Sequential(
    nn.Linear(64, 64),
    nn.Linear(64, 64)
)

In [43]:
# Assuming the model has been trained
torch.save(fp16_model.state_dict(), "model.pt")

In [44]:
# define an int8 model
int8_model = nn.Sequential(
    Linear8bitLt(64, 64, has_fp16_weights=False), # important to add the flag "has_fp16_weights"  
    Linear8bitLt(64, 64, has_fp16_weights=False)  # has_fp16_weights = True: 혼합 정밀도(mixed precision),Int8와 FP16을 섞어서 학습(training) 할 때 사용
)

In [45]:
# before calling the .to function
int8_model[0].weight

Parameter containing:
Parameter(Int8Params([[-0.0941, -0.0200,  0.0109,  ...,  0.1100, -0.0103,  0.0559],
            [-0.0254, -0.1154, -0.0383,  ...,  0.0579,  0.1184, -0.0874],
            [ 0.0826, -0.0834, -0.1150,  ..., -0.0094, -0.0910, -0.0336],
            ...,
            [ 0.0335,  0.0066,  0.0700,  ..., -0.0038,  0.0575, -0.0169],
            [-0.1193,  0.0880, -0.0385,  ...,  0.1197, -0.1134,  0.0605],
            [-0.0798, -0.0510, -0.0571,  ...,  0.0249,  0.0314, -0.0132]]))

In [46]:
# load the model in 8-bit
int8_model.load_state_dict(torch.load("model.pt"))
int8_model = int8_model.to(0) # Quantization happens here

In [47]:
int8_model[0].weight

Parameter containing:
Parameter(Int8Params([[ 121,  -54,   27,  ...,   10,   43,   68],
            [ 116,  -71,  -15,  ..., -106,   98,  -46],
            [ -75,   61,   92,  ...,  -72,   84,   49],
            ...,
            [  57, -110,   22,  ...,    4,   69,   56],
            [ -35,   34,  -86,  ...,  -37,   65,   -1],
            [ -79,   90,  -67,  ...,  122,   62,  127]], device='cuda:0',
           dtype=torch.int8))

In [48]:
# how to retrieve the FP16 weight in order to perform the outlier MatMul in fp16?
(int8_model[0].weight.CB * int8_model[0].weight.SCB) / 127

tensor([[ 0.1182, -0.0525,  0.0264,  ...,  0.0097,  0.0415,  0.0664],
        [ 0.1133, -0.0690, -0.0147,  ..., -0.1027,  0.0945, -0.0449],
        [-0.0733,  0.0593,  0.0899,  ..., -0.0698,  0.0810,  0.0478],
        ...,
        [ 0.0557, -0.1069,  0.0215,  ...,  0.0039,  0.0666,  0.0547],
        [-0.0342,  0.0330, -0.0841,  ..., -0.0358,  0.0627, -0.0010],
        [-0.0772,  0.0875, -0.0655,  ...,  0.1182,  0.0598,  0.1240]],
       device='cuda:0')

In [49]:
# infer using the model
input_ = torch.randn((1, 64), dtype=torch.float16)
hidden_states = int8_model(input_.to(torch.device('cuda', 0)))

## accelerate

In [58]:
import torch.nn as nn
from accelerate import init_empty_weights

# nn.Sequential: 여러 레이어를 차례대로 묶어 간단한 모델을 만드는 PyTorch 도구
with init_empty_weights():
    # 원래 코드: nn.Sequential([nn.Linear(100000, 100000) for _ in range(1000)])
    # *로 리스트를 풀어서 넘겨야 함 (unpacking)
    nn.Sequential(*[nn.Linear(100000, 100000) for _ in range(1000)]) # This will take ~0 RAM!